In [3]:
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from datetime import datetime

In [ ]:
# "indir" is a custom input path, and "outdir" is a custom output path.
indir = '../../03-stratified_RankSumTest_1vsOthers/01_subclasses_in_neuron_1vsOthers'
outdir = './'
modification_ = '5hmC'
scale_down = 17

def get_segment_length(segment_):
    chrom_, start_, end_ = segment_.split("_")
    return int(end_) - int(start_)

In [ ]:
with open(f'{outdir}/{modification_}G_volcano_plot.log', 'wt') as log_f:
    log_f.write(f'{datetime.now()}\t Started reading, {modification_}\n')
    original_diff_df=ad.read_h5ad(f'{indir}/{modification_}G_neuron_frac_segment_diff.h5ad').to_df()
    # perserve only segements with lengths that are at least 200bp
    data_columns_1=pd.Series(original_diff_df.columns)
    data_columns_2=data_columns_1[data_columns_1.apply(get_segment_length)>=200]
    diff = original_diff_df[data_columns_2].to_numpy().reshape(-1)[::scale_down]
    pvalue = ad.read_h5ad(f'{indir}/{modification_}G_neuron_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad').to_df().to_numpy().reshape(-1)[::scale_down]
    p_pvalue_ = -np.log10(pvalue)
    plt.figure(figsize=(12,6))
    x = diff[~(np.isnan(diff) | np.isnan(p_pvalue_) | np.isinf(p_pvalue_))]
    y = p_pvalue_[~(np.isnan(diff) | np.isnan(p_pvalue_) | np.isinf(p_pvalue_))]
    x_and_y = np.vstack([x,y])
    log_f.write(f'{datetime.now()}\t Started kde, {modification_}\n')
    kde = gaussian_kde(x_and_y)
    z = kde(x_and_y)
    idx = z.argsort()
    x, y, z = x[idx], y[idx], z[idx]
    log_f.write(f'{datetime.now()}\t Started plotting, {modification_}\n')
    plt.scatter(x, y, c=z, marker=".", s=1, edgecolors='none')
    plt.xlim(-0.6, +0.6); plt.xlabel(f'diff of {modification_}G in segments from robust mean');
    plt.ylim(0, 10); plt.ylabel("-log10(adjusted p-value)"); plt.title(f'volcano plot of {modification_}G in neuronal subclasses')
    if modification_ == "5mC":
        plt.vlines([-0.3,0.3], 0, 10, colors="red", linestyles="dotted")
    elif modification_ == "5hmC":
        plt.vlines([-0.2,0.2], 0, 10, colors="red", linestyles="dotted")
    plt.hlines([-np.log10(0.05)], -0.6, 0.6, colors="red", linestyles="dotted")
    log_f.write(f'{datetime.now()}\t Started saving, {modification_}\n')
    plt.savefig(f'{outdir}/{modification_}G_volcano_plot.png',dpi=600)
    log_f.write(f'{datetime.now()}\t Finished saving, {modification_}\n')

/tmp/ipykernel_113974/1069957293.py:9: RuntimeWarning: divide by zero encountered in log10
  p_pvalue_ = -np.log10(pvalue)
